In [ ]:
!pip install plotly pandas numpy -q

In [ ]:
# Импорт библиотек
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'colab'
print("Библиотеки загружены!")

Библиотеки загружены!


In [ ]:
# Генерация банковского датасета (2 года кредитных заявок)
np.random.seed(42)
N = 2000
cities = ['Москва', 'Санкт-Петербург', 'Казань', 'Новосибирск', 'Екатеринбург']
df = pd.DataFrame({
    'client_id': range(1001, 1001 + N),
    'city': np.random.choice(cities, N),
    'age': np.random.randint(21, 70, N),
    'income': np.random.normal(50000, 20000, N).astype(int),
    'loan_amount': np.random.normal(200000, 80000, N).astype(int),
    'credit_score': np.random.normal(650, 100, N).astype(int),
    'default': np.random.choice([0, 1], N, p=[0.9, 0.1])
})
df['income'] = df['income'].clip(lower=0)
df['loan_amount'] = df['loan_amount'].clip(lower=0)

# Добавляем временной ряд для графика динамики
dates = pd.date_range('2024-01-01', periods=24, freq='ME')
monthly = pd.DataFrame({
    'month': dates,
    'total_issued': np.random.normal(5000000, 800000, 24).astype(int)
})
monthly['total_issued'] = monthly['total_issued'].clip(lower=0)

print("Данные готовы!")
print(f"Клиентов: {df.shape[0]}")
print(f"Месяцев: {monthly.shape[0]}")

Данные готовы!
Клиентов: 2000
Месяцев: 24


## Считаем KPI

Эти 3 цифры покажут руководителю общее состояние портфеля.

In [ ]:
#   Расчёт KPI
#   total_clients — количество уникальных клиентов
#   avg_income — средний доход по портфелю
#   default_rate — доля дефолтов в процентах

total_clients = df['client_id'].nunique()
avg_income = int(df['income'].mean())
default_rate = round(df['default'].mean() * 100, 1)

print(f"Всего клиентов: {total_clients}")
print(f"Средний доход: {avg_income:,} руб.")
print(f"Доля дефолтов: {default_rate}%")

Всего клиентов: 2000
Средний доход: 49,777 руб.
Доля дефолтов: 10.6%


In [ ]:
# BoxPlot доходов по городам
# Показывает медиану, квартили и выбросы

fig_box = go.Figure()
for city in df['city'].unique():
    fig_box.add_trace(go.Box(
        y=df[df['city'] == city]['income'],
        name=city
    ))
fig_box.update_layout(
    title='Распределение доходов по городам',
    yaxis_title='Доход, руб.',
    height=400
)
fig_box.show()

In [ ]:
#   Scatter доход vs сумма кредита
#  Цвет показывает дефолт (0=синий, 1=красный)

fig_scatter = go.Figure()
fig_scatter.add_trace(go.Scatter(
    x=df['income'],
    y=df['loan_amount'],
    mode='markers',
    marker=dict(
        color=df['default'],
        colorscale=[[0, 'blue'], [1, 'red']],
        showscale=True,
        colorbar=dict(title='Дефолт')
    ),
    text=df['client_id'],
    name='Клиенты'
))
fig_scatter.update_layout(
    title='Доход vs Сумма кредита (цвет — дефолт)',
    xaxis_title='Доход, руб.',
    yaxis_title='Сумма кредита, руб.',
    height=400
)
fig_scatter.show()

In [ ]:
# Линия динамики выдач по месяцам

fig_line = go.Figure()
fig_line.add_trace(go.Scatter(
    x=monthly['month'],
    y=monthly['total_issued'],
    mode='lines+markers',
    line=dict(color='green', width=2),
    marker=dict(size=6),
    name='Объём выдач'
))
fig_line.update_layout(
    title='Динамика выдачи кредитов по месяцам',
    xaxis_title='Месяц',
    yaxis_title='Сумма выдач, руб.',
    height=400
)
fig_line.show()

In [ ]:
# Bar доли дефолтов по городам

default_by_city = df.groupby('city')['default'].mean() * 100
default_by_city = default_by_city.reset_index()
default_by_city.columns = ['city', 'default_rate']

fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(
    x=default_by_city['city'],
    y=default_by_city['default_rate'],
    marker_color='coral',
    text=default_by_city['default_rate'].round(1),
    textposition='outside',
    name='Доля дефолтов'
))
fig_bar.update_layout(
    title='Доля дефолтов по городам',
    yaxis_title='Доля дефолтов, %',
    height=400
)
fig_bar.show()

## Собираем дашборд — объединяем всё в одну панель

In [ ]:
#   make_subplots создаёт сетку 2x2
#   subplot_titles — заголовки для каждого блока
#   Добавляем 4 графика и KPI-аннотации


fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Распределение доходов по городам',
        'Доход vs Сумма кредита (цвет — дефолт)',
        'Динамика выдач по месяцам',
        'Доля дефолтов по городам'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.10
)

# Box (row=1, col=1)
for city in df['city'].unique():
    fig.add_trace(go.Box(
        y=df[df['city'] == city]['income'],
        name=city,
        showlegend=False
    ), row=1, col=1)

# Scatter (row=1, col=2)
fig.add_trace(go.Scatter(
    x=df['income'],
    y=df['loan_amount'],
    mode='markers',
    marker=dict(
        color=df['default'],
        colorscale=[[0, 'blue'], [1, 'red']],
        showscale=True,
        colorbar=dict(title='Дефолт', x=1.02, y=0.75, len=0.4)
    ),
    text=df['client_id'],
    name='Клиенты',
    showlegend=False
), row=1, col=2)

# Line (row=2, col=1)
fig.add_trace(go.Scatter(
    x=monthly['month'],
    y=monthly['total_issued'],
    mode='lines+markers',
    line=dict(color='green', width=2),
    marker=dict(size=6),
    name='Выдачи',
    showlegend=False
), row=2, col=1)

# Bar (row=2, col=2)
fig.add_trace(go.Bar(
    x=default_by_city['city'],
    y=default_by_city['default_rate'],
    marker_color='coral',
    text=default_by_city['default_rate'].round(1),
    textposition='outside',
    name='Дефолт %',
    showlegend=False
), row=2, col=2)

# KPI аннотации
kpi_text = f"👥 Клиентов: {total_clients}<br>💰 Средний доход: {avg_income:,} ₽<br>⚠️ Доля дефолтов: {default_rate}%"
fig.add_annotation(
    x=0.02, y=1.12,
    xref='paper', yref='paper',
    text=kpi_text,
    showarrow=False,
    font=dict(size=14, color='black'),
    align='left',
    bgcolor='lightyellow',
    bordercolor='black',
    borderwidth=1
)

# Финальные настройки
fig.update_layout(
    title_text='Кредитный портфель банка — дашборд',
    title_font_size=20,
    height=900,
    width=1100
)

fig.show()

## 🎸 Творческое задание

1. Поменяй цвета в scatter plot (вместо сине-красной схемы возьми 'viridis').
2. Добавь в дашборд пятый блок — гистограмму возраста клиентов.
3. Измени KPI: добавь общую сумму выданных кредитов.


In [ ]:
# твой код